# MLOps, Monitoring & Drift Detection
**Level:** Professional | **Interactive Machine Learning Curriculum**

> Model Registries, Data Drift vs Concept Drift, KS-tests, Population Stability Index (PSI), Feature Stores, and Automated Retraining.

---

### 📖 Conceptual Foundations

#### What is MLOps (Machine Learning Operations)?

**MLOps** is an engineering discipline that unifies Machine Learning system development (Dev) with system operations (Ops) to standardize and automate the continuous delivery of high-performing, reliable models into production. It addresses the reality that code represents only a tiny fraction (~5%) of real-world ML systems (Sculley et al., 'Hidden Technical Debt in Machine Learning Systems').


#### Types of Production Distribution Shift

| Shift Type | Mathematical Definition | Real-World Example | Recommended Diagnostic & Remedy |
| --- | --- | --- | --- |
| Covariate Shift (Data Drift) | $P(X)$ changes while $P(Y \mid X)$ remains constant. | Demographic user base expands from 20-30 to 50-60 year olds, but purchasing behavior for a given age is unchanged. | Monitor feature distributions using Kolmogorov-Smirnov (KS) tests or PSI; retrain on fresh covariate samples. |
| Concept Drift | $P(Y \mid X)$ changes while $P(X)$ remains constant. | Macroeconomic inflation changes customer spending habits; high-income users suddenly buy budget products. | Track rolling validation loss and ground-truth error rates; update labels and retrain model architecture. |
| Prior Probability Shift | $P(Y)$ changes while $P(X \mid Y)$ remains constant. | Sudden seasonal spike in flu diagnosis during winter months. | Adjust model decision thresholds or update class prior weights in the loss function. |


#### Population Stability Index (PSI) Formula

The **Population Stability Index (PSI)** measures the difference between an initial baseline (reference) distribution $B$ and a live production (target) distribution $T$ divided into $K$ bins:

$$\text{PSI} = \sum_{i=1}^{K} (T_i - B_i) \times \ln\left( \frac{T_i}{B_i} \right)$$

- **$\text{PSI} < 0.10$:** Negligible shift; no action required.
- **$0.10 \le \text{PSI} < 0.25$:** Moderate drift; investigate features and plan retraining.
- **$\text{PSI} \ge 0.25$:** Significant drift! High risk of performance collapse; trigger immediate retraining or fallback.


#### The Feature Store Architecture (e.g., Feast, Tecton)

In production, feature pipelines often suffer from **Training-Serving Skew** (features computed differently in offline Spark jobs vs online real-time APIs). Feature stores decouple feature engineering:
1. **Offline Store (BigQuery, Snowflake, S3):** Optimized for high-throughput batch queries across millions of historical records during model training.
2. **Online Store (Redis, DynamoDB):** Ultra-low latency (<5ms) key-value lookup for real-time feature retrieval during inference.
3. **Point-in-Time Correctness (Time Travel):** Guarantees zero future data leakage when generating historical training features.



In [ ]:
# %matplotlib inline
import numpy as np

def calculate_psi(expected, actual, num_bins=10):
    # Divide expected distribution into quantile bins
    quantiles = np.linspace(0, 100, num_bins + 1)
    bin_edges = np.percentile(expected, quantiles)
    bin_edges[0] -= 1e-5
    bin_edges[-1] += 1e-5
    
    # Count frequencies
    expected_counts, _ = np.histogram(expected, bins=bin_edges)
    actual_counts, _ = np.histogram(actual, bins=bin_edges)
    
    # Convert to proportions with smoothing epsilon to prevent log(0)
    e_pct = (expected_counts + 1e-4) / (len(expected) + 1e-4 * num_bins)
    a_pct = (actual_counts + 1e-4) / (len(actual) + 1e-4 * num_bins)
    
    # Calculate PSI
    psi_val = np.sum((a_pct - e_pct) * np.log(a_pct / e_pct))
    return psi_val

# Baseline reference data ~ N(0, 1)
np.random.seed(42)
reference = np.random.randn(5000)

# Case 1: Live traffic without drift ~ N(0.02, 1.01)
live_stable = np.random.randn(2000) * 1.01 + 0.02
psi_stable = calculate_psi(reference, live_stable)

# Case 2: Live traffic with significant covariate shift ~ N(0.6, 1.3)
live_drifted = np.random.randn(2000) * 1.3 + 0.6
psi_drifted = calculate_psi(reference, live_drifted)

print(f'Stable Traffic PSI:  {psi_stable:.4f} -> Evaluation: {"Healthy (<0.1)" if psi_stable < 0.1 else "Drifted"}')
print(f'Drifted Traffic PSI: {psi_drifted:.4f} -> Evaluation: {"CRITICAL DRIFT (>=0.25)" if psi_drifted >= 0.25 else "Moderate"}')


---

### 🎯 Hands-On Experiments & Challenges
1. **Experiment with parameters:** Modify the hyperparameters or dataset sizes in the cell above.
2. **Add visual diagnostics:** Plot loss curves or feature distributions using Matplotlib/Seaborn.
3. **Test with new data:** Try running the model on synthetic noisy data or custom inputs.


In [ ]:
# Your custom experiment code here:

